<a href="https://colab.research.google.com/github/Arslan1Asim/Flyrank-ML-Internship/blob/main/ML_Capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML Capstone — Refresh / Content Opportunity Scoring

## 1. Problem Definition

### Decision
Which content should a team prioritize for review, refresh, optimization, promotion, or monitoring?

### Unit of Analysis
One content item at a defined observation point.

### Prediction Moment
The model makes its prediction using information available up to the end of the observation window.

### Target
Predict whether a content item will show a meaningful decline in the subsequent outcome window.

### Practical Goal
Produce a ranked list of content items that deserve attention, together with the signals/reasons supporting each recommendation.

### ML Task
Supervised binary classification followed by ranking/scoring.

### Evaluation Goal
Determine whether the ML model provides useful prioritization beyond a simple rule-based baseline, using a leakage-aware validation design.

## 2. Prediction Timeline

We separate the data into two conceptual windows:

- Observation window: historical information available when the prediction is made.
- Outcome window: the future period used to determine whether the content subsequently declined.

Features must be constructed only from information available during the observation window.

The outcome window must not be used to construct predictors.

## 3. Data Discovery

The capstone uses the full FlyRank internship warehouse hosted on Hugging Face.

The warehouse is accessed through DuckDB rather than downloaded in full. The relevant tables and fields will be inspected before feature engineering so that the analysis is based on the actual available data.

The goal of this step is to identify:
- the relevant content and performance tables,
- the unit of analysis,
- available date ranges,
- candidate outcome variables,
- candidate predictor variables,
- and variables that may create leakage.

In [3]:
import os
import getpass

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

In [6]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')

dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


In [3]:
# Inspect the schemas of the main tables for the capstone

print("=== DIM_CONTENT ===")
display(con.sql(f"""
    DESCRIBE SELECT * FROM {TABLES['dim_content']}
""").df())

print("=== FACT_DAILY ===")
display(con.sql(f"""
    DESCRIBE SELECT * FROM {TABLES['fact_daily']}
""").df())

print("=== FACT_QUERY_90D ===")
display(con.sql(f"""
    DESCRIBE SELECT * FROM {TABLES['fact_query_90d']}
""").df())

=== DIM_CONTENT ===


,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,content_hash_id,VARCHAR,YES,None,None,None
2,keyword_hash_id,VARCHAR,YES,None,None,None
3,url_hash_id,VARCHAR,YES,None,None,None
4,keyword_char_count,BIGINT,YES,None,None,None
5,keyword_token_count,BIGINT,YES,None,None,None
6,url_char_count,BIGINT,YES,None,None,None
7,content_created_date,DATE,YES,None,None,None
8,content_updated_date,DATE,YES,None,None,None
9,content_type,VARCHAR,YES,None,None,None


=== FACT_DAILY ===


,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


=== FACT_QUERY_90D ===


,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,content_hash_id,VARCHAR,YES,None,None,None
2,query_hash_id,VARCHAR,YES,None,None,None
3,query_char_count,BIGINT,YES,None,None,None
4,query_token_count,BIGINT,YES,None,None,None
5,window_start,DATE,YES,None,None,None
6,window_end,DATE,YES,None,None,None
7,impressions_90d,BIGINT,YES,None,None,None
8,clicks_90d,BIGINT,YES,None,None,None
9,impressions_last30,BIGINT,YES,None,None,None


## 4. Data Coverage and History

Before defining the target, we inspect the temporal coverage and history available for each content item.

The goal is to determine:
- the overall reporting period,
- how many content items have usable historical data,
- how much data is available per content item,
- and whether there is sufficient future data to construct a leakage-free outcome window.

In [4]:
# Overall date coverage and row counts

coverage = con.sql(f"""
    SELECT
        MIN(report_date) AS earliest_date,
        MAX(report_date) AS latest_date,
        COUNT(*) AS total_rows,
        COUNT(DISTINCT content_hash_id) AS unique_content_items,
        COUNT(DISTINCT client_hash_id) AS unique_clients
    FROM {TABLES['fact_daily']}
""").df()

display(coverage)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,earliest_date,latest_date,total_rows,unique_content_items,unique_clients
0,2025-01-27,2026-06-30,78835655,427292,70


In [5]:
# Distribution of the number of daily observations per content item

history = con.sql(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        COUNT(*) AS observation_days,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM {TABLES['fact_daily']}
    GROUP BY client_hash_id, content_hash_id
""").df()

display(history['observation_days'].describe())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,observation_days
count,427292.000000
mean,184.500658
std,93.721777
min,3.000000
25%,112.000000
50%,218.000000
75%,240.000000
max,520.000000


In [6]:
con.sql(f"""
SELECT
    c.gsc_data_start,
    COUNT(DISTINCT f.content_hash_id) AS content_items,
    MIN(f.report_date) AS earliest_obs,
    MAX(f.report_date) AS latest_obs
FROM {TABLES['fact_daily']} f
JOIN {TABLES['dim_clients']} c USING (client_hash_id)
GROUP BY c.gsc_data_start
ORDER BY c.gsc_data_start
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,gsc_data_start,content_items,earliest_obs,latest_obs
0,2025-01-27,7598,2025-01-27,2026-06-30
1,2025-02-11,30983,2025-02-11,2026-06-30
2,2025-03-11,12267,2025-03-11,2026-06-30
3,2025-06-07,27872,2025-06-07,2026-06-30
4,2025-06-18,5032,2025-06-18,2026-06-30
5,2025-06-21,15783,2025-06-21,2026-06-30
6,2025-06-29,11193,2025-06-29,2026-06-30
7,2025-07-01,31887,2025-07-01,2026-06-30
8,2025-07-06,2618,2025-07-06,2026-06-30
9,2025-07-07,2063,2025-07-07,2026-06-30


In [7]:
con.sql(f"""
SELECT
    f.client_hash_id,
    c.gsc_data_start,
    MIN(f.report_date) AS actual_earliest,
    MAX(f.report_date) AS actual_latest,
    COUNT(DISTINCT f.content_hash_id) AS content_items
FROM {TABLES['fact_daily']} f
LEFT JOIN {TABLES['dim_clients']} c USING (client_hash_id)
GROUP BY f.client_hash_id, c.gsc_data_start
HAVING actual_earliest < c.gsc_data_start OR c.gsc_data_start IS NULL
ORDER BY actual_earliest
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,gsc_data_start,actual_earliest,actual_latest,content_items
0,client_e547b89c05043229,2025-11-15,2025-10-13,2026-06-30,9639
1,client_19b89ee4fe3db6da,NaT,2025-10-13,2026-06-25,6871
2,client_400c21c81c8b46ef,2025-10-26,2025-10-13,2026-06-30,4744
3,client_2094c6eb080311d5,2025-12-17,2025-10-13,2026-06-30,8445
4,client_a60a11451483af1c,2025-11-16,2025-11-05,2026-06-29,5655
5,client_f623b01661d4bfe4,2026-01-09,2025-11-05,2026-06-25,5878
6,client_764ae36a94e30a25,2025-11-07,2025-11-05,2026-05-31,121
7,client_770e8e5faa9cddfe,2025-11-17,2025-11-05,2026-06-30,200
8,client_80ee5b7bd5f4eb89,NaT,2025-11-05,2026-06-30,196
9,client_ccdd78843409c8c7,2026-02-17,2026-02-12,2026-06-30,417


In [8]:
con.sql(f"""
SELECT report_date, content_hash_id, gsc_impressions, gsc_clicks, gsc_avg_position
FROM {TABLES['fact_daily']} -- Correctly reference the parquet source
WHERE client_hash_id = 'client_2094c6eb080311d5'
  AND report_date < '2025-12-17'
ORDER BY report_date
LIMIT 20
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position
0,2025-10-13,content_7c3f47a472306d35,0,0,NaN
1,2025-10-13,content_c473d26fde97d2db,0,0,NaN
2,2025-10-13,content_c825a86c474156ba,0,0,NaN
3,2025-10-13,content_bc270b4a64b2f2bc,0,0,NaN
4,2025-10-13,content_36372acb89bf8005,0,0,NaN
5,2025-10-13,content_1e8ee99ca4d1844c,0,0,NaN
6,2025-10-13,content_0cb59a8985c954bf,0,0,NaN
7,2025-10-13,content_49bad31f31a25726,0,0,NaN
8,2025-10-13,content_a66b63255159b723,0,0,NaN
9,2025-10-13,content_842227a43549245b,0,0,NaN


In [9]:
con.sql(f"""
SELECT report_date,
       COUNT(*) AS rows,
       SUM(gsc_impressions) AS total_impressions,
       SUM(CASE WHEN gsc_impressions > 0 THEN 1 ELSE 0 END) AS nonzero_rows
FROM {TABLES['fact_daily']}
WHERE client_hash_id = 'client_2094c6eb080311d5'
  AND report_date BETWEEN '2025-10-13' AND '2025-12-20'
GROUP BY report_date
ORDER BY report_date
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,rows,total_impressions,nonzero_rows
0,2025-10-13,1869,0.0,0.0
1,2025-10-14,1869,0.0,0.0
2,2025-10-15,1869,0.0,0.0
3,2025-10-16,2319,0.0,0.0
4,2025-10-17,2319,0.0,0.0
...,...,...,...,...
64,2025-12-16,2319,0.0,0.0
65,2025-12-17,2319,1336.0,440.0
66,2025-12-18,2319,1019.0,300.0
67,2025-12-19,2319,954.0,185.0


In [10]:
con.sql(f"""
SELECT report_date, SUM(gsc_impressions) AS total_impressions
FROM {TABLES['fact_daily']}
WHERE client_hash_id = 'client_19b89ee4fe3db6da'
GROUP BY report_date
ORDER BY report_date
LIMIT 30
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,total_impressions
0,2025-10-13,0.0
1,2025-10-14,0.0
2,2025-10-15,0.0
3,2025-10-16,0.0
4,2025-10-17,0.0
5,2025-10-18,0.0
6,2025-10-19,0.0
7,2025-10-20,0.0
8,2025-10-21,0.0
9,2025-10-22,0.0


In [11]:
con.sql(f"""
SELECT MIN(report_date) AS first_nonzero
FROM {TABLES['fact_daily']}
WHERE client_hash_id = 'client_19b89ee4fe3db6da'
  AND gsc_impressions > 0
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,first_nonzero
0,NaT


In [12]:
con.sql(f"""
SELECT client_hash_id,
       COUNT(*) AS total_rows,
       SUM(gsc_impressions) AS total_impressions,
       MIN(report_date) FILTER (WHERE gsc_impressions > 0) AS first_nonzero,
       MAX(report_date) AS last_date
FROM {TABLES['fact_daily']}
WHERE client_hash_id IN (
    'client_19b89ee4fe3db6da',
    'client_80ee5b7bd5f4eb89',
    'client_04660893ae39614a'
)
GROUP BY client_hash_id
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,total_rows,total_impressions,first_nonzero,last_date
0,client_19b89ee4fe3db6da,1653823,0.0,NaT,2026-06-25
1,client_80ee5b7bd5f4eb89,46648,0.0,NaT,2026-06-30
2,client_04660893ae39614a,36039,0.0,NaT,2026-06-30


In [13]:
con.sql(f"""
SELECT content_hash_id, COUNT(*) AS n_windows,
       MIN(window_end) AS earliest_window, MAX(window_end) AS latest_window
FROM {TABLES['fact_query_90d']}
GROUP BY content_hash_id
ORDER BY n_windows DESC
LIMIT 10
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,n_windows,earliest_window,latest_window
0,content_eadb33b5df496f4a,7889,2026-06-30,2026-06-30
1,content_545bb6cc7081ded3,3378,2026-06-30,2026-06-30
2,content_0e03de7680314cd5,2241,2026-06-30,2026-06-30
3,content_9ef3d7516483e665,1577,2026-06-30,2026-06-30
4,content_661a7734f691bef5,1551,2026-06-30,2026-06-30
5,content_8d7d99f109e19aa2,1399,2026-06-30,2026-06-30
6,content_93a9b8328d4fd032,1358,2026-06-30,2026-06-30
7,content_33d31496fca9665e,1335,2026-06-30,2026-06-30
8,content_77276ad7a26f4905,1294,2026-06-30,2026-06-30
9,content_61215c724c8220ae,1285,2026-06-30,2026-06-30


In [14]:
con.sql(f"DESCRIBE SELECT * FROM {TABLES['dim_content']}").df().to_string()

'                   column_name column_type null   key default extra\n0               client_hash_id     VARCHAR  YES  None    None  None\n1              content_hash_id     VARCHAR  YES  None    None  None\n2              keyword_hash_id     VARCHAR  YES  None    None  None\n3                  url_hash_id     VARCHAR  YES  None    None  None\n4           keyword_char_count      BIGINT  YES  None    None  None\n5          keyword_token_count      BIGINT  YES  None    None  None\n6               url_char_count      BIGINT  YES  None    None  None\n7         content_created_date        DATE  YES  None    None  None\n8         content_updated_date        DATE  YES  None    None  None\n9                 content_type     VARCHAR  YES  None    None  None\n10               search_volume      BIGINT  YES  None    None  None\n11                 competition      DOUBLE  YES  None    None  None\n12           competition_level     VARCHAR  YES  None    None  None\n13                         cpc   

In [15]:
con.sql(f"""
SELECT
    MIN(window_start) AS min_window_start,
    MAX(window_start) AS max_window_start,
    MIN(window_end) AS min_window_end,
    MAX(window_end) AS max_window_end,
    COUNT(DISTINCT window_end) AS distinct_window_ends
FROM {TABLES['fact_query_90d']}
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,min_window_start,max_window_start,min_window_end,max_window_end,distinct_window_ends
0,2026-04-02,2026-04-02,2026-06-30,2026-06-30,1


In [16]:
import pandas as pd
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

desc = con.sql(f"DESCRIBE SELECT * FROM {TABLES['dim_content']}").df()
print(desc.to_string())

                   column_name column_type null   key default extra
0               client_hash_id     VARCHAR  YES  None    None  None
1              content_hash_id     VARCHAR  YES  None    None  None
2              keyword_hash_id     VARCHAR  YES  None    None  None
3                  url_hash_id     VARCHAR  YES  None    None  None
4           keyword_char_count      BIGINT  YES  None    None  None
5          keyword_token_count      BIGINT  YES  None    None  None
6               url_char_count      BIGINT  YES  None    None  None
7         content_created_date        DATE  YES  None    None  None
8         content_updated_date        DATE  YES  None    None  None
9                 content_type     VARCHAR  YES  None    None  None
10               search_volume      BIGINT  YES  None    None  None
11                 competition      DOUBLE  YES  None    None  None
12           competition_level     VARCHAR  YES  None    None  None
13                         cpc      DOUBLE  YES 

In [17]:
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(optimization_eligible_date) AS has_elig_date,
    COUNT(last_optimized_date) AS has_optimized_date,
    COUNT(content_updated_date) AS has_updated_date,
    MIN(optimization_eligible_date) AS elig_min, MAX(optimization_eligible_date) AS elig_max,
    MIN(last_optimized_date) AS opt_min, MAX(last_optimized_date) AS opt_max,
    MIN(content_updated_date) AS upd_min, MAX(content_updated_date) AS upd_max,
    SUM(CASE WHEN is_published THEN 1 ELSE 0 END) AS n_published,
    SUM(CASE WHEN is_deleted THEN 1 ELSE 0 END) AS n_deleted
FROM {TABLES['dim_content']}
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,has_elig_date,has_optimized_date,has_updated_date,elig_min,elig_max,opt_min,opt_max,upd_min,upd_max,n_published,n_deleted
0,519606,45396,45396,519606,2026-06-08,2026-08-20,2026-04-24,2026-07-06,2024-10-28,2026-07-06,411540.0,101559.0


In [18]:
con.sql(f"""
SELECT
    optimization_eligible_date - last_optimized_date AS cooldown_days,
    COUNT(*) AS n
FROM {TABLES['dim_content']}
WHERE last_optimized_date IS NOT NULL
GROUP BY cooldown_days
ORDER BY n DESC
LIMIT 10
""").df()

,cooldown_days,n
0,45,45396


In [19]:
con.sql(f"""
SELECT
    is_published,
    is_deleted,
    COUNT(*) AS n
FROM {TABLES['dim_content']}
GROUP BY is_published, is_deleted
ORDER BY n DESC
""").df()

,is_published,is_deleted,n
0,True,False,411540
1,False,True,101559
2,False,False,6507


In [20]:
con.sql(f"""
SELECT
    dc.is_published,
    dc.is_deleted,
    COUNT(DISTINCT fd.content_hash_id) AS content_items_in_fact_daily
FROM {TABLES['fact_daily']} fd
JOIN {TABLES['dim_content']} dc USING (content_hash_id)
GROUP BY dc.is_published, dc.is_deleted
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,is_published,is_deleted,content_items_in_fact_daily
0,False,True,18897
1,True,False,402064
2,False,False,6331


In [21]:
con.sql(f"""
SELECT
    dc.is_published,
    dc.is_deleted,
    COUNT(DISTINCT fd.content_hash_id) AS content_items,
    SUM(CASE WHEN fd.gsc_impressions > 0 THEN 1 ELSE 0 END) AS nonzero_rows,
    COUNT(DISTINCT CASE WHEN fd.gsc_impressions > 0 THEN fd.content_hash_id END) AS items_with_any_signal,
    MIN(fd.report_date) FILTER (WHERE fd.gsc_impressions > 0) AS earliest_signal,
    MAX(fd.report_date) FILTER (WHERE fd.gsc_impressions > 0) AS latest_signal
FROM {TABLES['fact_daily']} fd
JOIN {TABLES['dim_content']} dc USING (content_hash_id)
GROUP BY dc.is_published, dc.is_deleted
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,is_published,is_deleted,content_items,nonzero_rows,items_with_any_signal,earliest_signal,latest_signal
0,True,False,402064,28779788.0,292949,2025-01-27,2026-06-30
1,False,False,6331,11889.0,4004,2025-09-24,2026-06-30
2,False,True,18897,178374.0,12281,2025-06-26,2026-06-30


In [22]:
con.sql(f"""
WITH client_splits AS (
    SELECT c.client_hash_id, c.gsc_data_start,
           MAX(f.report_date) AS latest_obs,
           c.gsc_data_start + INTERVAL (CAST(DATE_DIFF('day', c.gsc_data_start, MAX(f.report_date)) * 0.8 AS INT)) DAY AS split_date
    FROM {TABLES['fact_daily']} f
    JOIN {TABLES['dim_clients']} c USING (client_hash_id)
    WHERE c.client_hash_id NOT IN ('client_19b89ee4fe3db6da','client_80ee5b7bd5f4eb89','client_04660893ae39614a')
    GROUP BY c.client_hash_id, c.gsc_data_start
)
SELECT
    f.content_hash_id,
    COUNT(*) FILTER (WHERE f.gsc_impressions > 0) AS active_days_in_obs_window
FROM {TABLES['fact_daily']} f
JOIN client_splits s ON f.client_hash_id = s.client_hash_id
WHERE f.report_date BETWEEN s.gsc_data_start AND s.split_date
GROUP BY f.content_hash_id
""").df()['active_days_in_obs_window'].describe()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,active_days_in_obs_window
count,395931.000000
mean,51.773534
std,79.919983
min,0.000000
25%,0.000000
50%,12.000000
75%,73.000000
max,416.000000


In [23]:
con.sql(f"""
WITH client_splits AS (
    SELECT c.client_hash_id, c.gsc_data_start,
           MAX(f.report_date) AS latest_obs,
           c.gsc_data_start + INTERVAL (CAST(DATE_DIFF('day', c.gsc_data_start, MAX(f.report_date)) * 0.8 AS INT)) DAY AS split_date
    FROM {TABLES['fact_daily']} f
    JOIN {TABLES['dim_clients']} c USING (client_hash_id)
    WHERE c.client_hash_id NOT IN ('client_19b89ee4fe3db6da','client_80ee5b7bd5f4eb89','client_04660893ae39614a')
    GROUP BY c.client_hash_id, c.gsc_data_start
),
active_days AS (
    SELECT f.content_hash_id,
           COUNT(*) FILTER (WHERE f.gsc_impressions > 0) AS active_days
    FROM {TABLES['fact_daily']} f
    JOIN client_splits s ON f.client_hash_id = s.client_hash_id
    WHERE f.report_date BETWEEN s.gsc_data_start AND s.split_date
    GROUP BY f.content_hash_id
)
SELECT
    SUM(CASE WHEN active_days >= 14 THEN 1 ELSE 0 END) AS floor_14,
    SUM(CASE WHEN active_days >= 21 THEN 1 ELSE 0 END) AS floor_21,
    SUM(CASE WHEN active_days >= 30 THEN 1 ELSE 0 END) AS floor_30,
    SUM(CASE WHEN active_days >= 45 THEN 1 ELSE 0 END) AS floor_45,
    SUM(CASE WHEN active_days >= 60 THEN 1 ELSE 0 END) AS floor_60,
    COUNT(*) AS total
FROM active_days
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,floor_14,floor_21,floor_30,floor_45,floor_60,total
0,191906.0,174026.0,155175.0,131161.0,111313.0,395931


## 5. Prediction Design

### Observation Window

The model will use the 30 days immediately preceding the prediction point to construct predictor variables.

### Prediction Point

The prediction is made at the end of the 30-day observation window.

### Outcome Window

The following 30 days are reserved for measuring subsequent content performance and defining the outcome.

### Leakage Rule

No information from the outcome window may be used to construct predictor variables.

The prediction design therefore separates information available at decision time from information that becomes known only after the prediction is made.

## 6. Target Exploration

Before defining the final decline label, we examine how content performance changes between consecutive 30-day windows.

The purpose is to understand the distribution of performance change before selecting a threshold for a meaningful decline.

In [24]:
# Explore performance change between two consecutive 30-day windows.
# This is exploratory only; the final target has not been defined yet.

target_exploration = con.sql(f"""
WITH date_bounds AS (
    SELECT MAX(report_date) AS latest_date
    FROM {TABLES['fact_daily']}
),

daily AS (
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        COALESCE(gsc_impressions, 0) AS impressions,
        COALESCE(gsc_clicks, 0) AS clicks
    FROM {TABLES['fact_daily']}
),

windowed AS (
    SELECT
        d.client_hash_id,
        d.content_hash_id,

        SUM(
            CASE
                WHEN d.report_date > b.latest_date - INTERVAL '60 days'
                 AND d.report_date <= b.latest_date - INTERVAL '30 days'
                THEN d.impressions ELSE 0
            END
        ) AS observation_impressions,

        SUM(
            CASE
                WHEN d.report_date > b.latest_date - INTERVAL '30 days'
                 AND d.report_date <= b.latest_date
                THEN d.impressions ELSE 0
            END
        ) AS outcome_impressions

    FROM daily d
    CROSS JOIN date_bounds b
    GROUP BY
        d.client_hash_id,
        d.content_hash_id
)

SELECT
    observation_impressions,
    outcome_impressions,
    CASE
        WHEN observation_impressions > 0
        THEN (outcome_impressions - observation_impressions)
             / CAST(observation_impressions AS DOUBLE)
        ELSE NULL
    END AS relative_change
FROM windowed
WHERE observation_impressions > 0
  AND outcome_impressions IS NOT NULL
""").df()

display(target_exploration.describe())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,observation_impressions,outcome_impressions,relative_change
count,237429.000000,237429.000000,237429.000000
mean,1084.880545,875.167389,0.190208
std,4884.918274,4723.680003,17.274843
min,1.000000,0.000000,-1.000000
25%,8.000000,1.000000,-0.944921
50%,78.000000,38.000000,-0.556848
75%,497.000000,355.000000,-0.109195
max,585502.000000,615012.000000,6511.000000


## 7. Target Sensitivity Analysis

The raw relative-change distribution is heavily influenced by content with very low observation-period impressions.

Before selecting the final decline threshold, we therefore examine how the distribution changes when low-volume content is excluded.

This helps ensure that the target represents a meaningful performance decline rather than random movement among very low-volume content.

In [25]:
# Examine decline distributions at different minimum observation volumes.

threshold_analysis = con.sql("""
SELECT
    '>= 10 impressions' AS volume_threshold,
    COUNT(*) AS content_items,
    AVG(relative_change) AS mean_change,
    MEDIAN(relative_change) AS median_change,
    QUANTILE_CONT(relative_change, 0.25) AS p25,
    QUANTILE_CONT(relative_change, 0.75) AS p75
FROM target_exploration
WHERE observation_impressions >= 10

UNION ALL

SELECT
    '>= 30 impressions',
    COUNT(*),
    AVG(relative_change),
    MEDIAN(relative_change),
    QUANTILE_CONT(relative_change, 0.25),
    QUANTILE_CONT(relative_change, 0.75)
FROM target_exploration
WHERE observation_impressions >= 30

UNION ALL

SELECT
    '>= 100 impressions',
    COUNT(*),
    AVG(relative_change),
    MEDIAN(relative_change),
    QUANTILE_CONT(relative_change, 0.25),
    QUANTILE_CONT(relative_change, 0.75)
FROM target_exploration
WHERE observation_impressions >= 100

UNION ALL

SELECT
    '>= 500 impressions',
    COUNT(*),
    AVG(relative_change),
    MEDIAN(relative_change),
    QUANTILE_CONT(relative_change, 0.25),
    QUANTILE_CONT(relative_change, 0.75)
FROM target_exploration
WHERE observation_impressions >= 500
""").df()

display(threshold_analysis)

,volume_threshold,content_items,mean_change,median_change,p25,p75
0,>= 10 impressions,173345,0.039993,-0.457627,-0.750000,-0.027027
1,>= 30 impressions,146961,-0.011852,-0.421053,-0.695958,-0.004440
2,>= 100 impressions,111247,-0.073278,-0.388535,-0.641787,0.001071
3,>= 500 impressions,59196,-0.154222,-0.372510,-0.602236,-0.033728


## 8. Candidate Target Thresholds

We evaluate several candidate definitions of a meaningful decline.

Only content with at least 100 impressions in the observation window is considered, reducing the influence of extremely low-volume content.

Candidate thresholds of 20%, 30%, 40%, and 50% decline are compared to understand the resulting class balance before selecting the final target definition.

In [26]:
# Compare candidate decline thresholds.

candidate_thresholds = con.sql("""
SELECT
    threshold,
    COUNT(*) AS eligible_items,
    SUM(CASE WHEN relative_change <= -threshold THEN 1 ELSE 0 END) AS declining_items,
    ROUND(
        100.0 * SUM(CASE WHEN relative_change <= -threshold THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS decline_rate_pct
FROM (
    SELECT 0.20 AS threshold
    UNION ALL SELECT 0.30
    UNION ALL SELECT 0.40
    UNION ALL SELECT 0.50
) thresholds
CROSS JOIN target_exploration
WHERE observation_impressions >= 100
GROUP BY threshold
ORDER BY threshold
""").df()

display(candidate_thresholds)

,threshold,eligible_items,declining_items,decline_rate_pct
0,0.2,111247,71592.0,64.35
1,0.3,111247,63611.0,57.18
2,0.4,111247,54479.0,48.97
3,0.5,111247,43916.0,39.48


## 9. Final Target Definition

A content item is labeled as declining (`is_declining_label = 1`) when clicks in the subsequent 30-day outcome window are at least 40% lower than clicks in the preceding 30-day observation window.

Eligibility requirement:
- Observation-window impressions >= 100

Target construction:
- Observation window: 30 days
- Outcome window: following 30 days
- Decline threshold: 40%

This definition is designed to reduce low-volume noise while maintaining a balanced target distribution.

## 10. Historical Observation-Outcome Windows

Construct sequential 30-day observation and 30-day outcome windows from the historical data.

Each training example must follow:

30-day observation window → 30-day outcome window → decline label

The observation window is used to calculate predictor features, while the following outcome window is used only to create the target label.

Eligibility requires at least 100 impressions during the observation window.

In [27]:
con.sql(f"""DESCRIBE SELECT * FROM {TABLES['fact_daily']}""").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 11. Historical Observation Outcome Windows

In [28]:
historical_windows = con.sql(f"""
WITH daily AS (
    SELECT
        content_hash_id,
        report_date,
        gsc_impressions,
        gsc_clicks
    FROM {TABLES['fact_daily']}
    WHERE gsc_data_available = TRUE
),

windowed AS (
    SELECT
        content_hash_id,
        report_date,

        -- Current 30-day observation window
        SUM(gsc_impressions) OVER (
            PARTITION BY content_hash_id
            ORDER BY report_date
            RANGE BETWEEN INTERVAL 29 DAY PRECEDING AND CURRENT ROW
        ) AS observation_impressions,

        SUM(gsc_clicks) OVER (
            PARTITION BY content_hash_id
            ORDER BY report_date
            RANGE BETWEEN INTERVAL 29 DAY PRECEDING AND CURRENT ROW
        ) AS observation_clicks,

        -- Following 30-day outcome window
        SUM(gsc_impressions) OVER (
            PARTITION BY content_hash_id
            ORDER BY report_date
            RANGE BETWEEN CURRENT ROW AND INTERVAL 30 DAY FOLLOWING
        ) AS outcome_impressions,

        SUM(gsc_clicks) OVER (
            PARTITION BY content_hash_id
            ORDER BY report_date
            RANGE BETWEEN CURRENT ROW AND INTERVAL 30 DAY FOLLOWING
        ) AS outcome_clicks

    FROM daily
)

SELECT
    content_hash_id,

    report_date - INTERVAL 29 DAY AS observation_start,
    report_date AS observation_end,

    report_date + INTERVAL 1 DAY AS outcome_start,
    report_date + INTERVAL 30 DAY AS outcome_end,

    observation_impressions,
    observation_clicks,
    outcome_impressions,
    outcome_clicks,

    CASE
        WHEN observation_impressions >= 100
         AND observation_clicks > 0
         AND outcome_clicks <= observation_clicks * 0.60
        THEN 1
        ELSE 0
    END AS is_declining_label

FROM windowed

WHERE observation_impressions >= 100
  AND report_date + INTERVAL 30 DAY <= (
      SELECT MAX(report_date)
      FROM daily
  )

ORDER BY content_hash_id, report_date
""").df()

historical_windows.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,observation_start,observation_end,outcome_start,outcome_end,observation_impressions,observation_clicks,outcome_impressions,outcome_clicks,is_declining_label
0,content_000005d4ced12088,2025-03-29,2025-04-27,2025-04-28,2025-05-27,115.0,1.0,276.0,0.0,1
1,content_000005d4ced12088,2025-03-30,2025-04-28,2025-04-29,2025-05-28,126.0,1.0,268.0,0.0,1
2,content_000005d4ced12088,2025-03-31,2025-04-29,2025-04-30,2025-05-29,140.0,1.0,261.0,0.0,1
3,content_000005d4ced12088,2025-04-01,2025-04-30,2025-05-01,2025-05-30,146.0,1.0,257.0,0.0,1
4,content_000005d4ced12088,2025-04-02,2025-05-01,2025-05-02,2025-05-31,146.0,1.0,257.0,0.0,1


In [29]:
con.sql(f"""
SELECT
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date,
    COUNT(*) AS rows
FROM {TABLES['fact_daily']}
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,min_date,max_date,rows
0,2025-01-27,2026-06-30,78835655


In [30]:
print("Total historical windows:", len(historical_windows))

print("\nTarget distribution:")
print(
    historical_windows["is_declining_label"]
    .value_counts()
    .sort_index()
)

print("\nTarget distribution (%):")
print(
    historical_windows["is_declining_label"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

Total historical windows: 18284739

Target distribution:
is_declining_label
0    14201077
1     4083662
Name: count, dtype: int64

Target distribution (%):
is_declining_label
0    77.67
1    22.33
Name: proportion, dtype: float64


In [31]:
historical_windows[
    [
        "observation_start",
        "observation_end",
        "outcome_start",
        "outcome_end",
        "observation_impressions",
        "observation_clicks",
        "outcome_impressions",
        "outcome_clicks",
        "is_declining_label"
    ]
].head(10)

,observation_start,observation_end,outcome_start,outcome_end,observation_impressions,observation_clicks,outcome_impressions,outcome_clicks,is_declining_label
0,2025-03-29,2025-04-27,2025-04-28,2025-05-27,115.0,1.0,276.0,0.0,1
1,2025-03-30,2025-04-28,2025-04-29,2025-05-28,126.0,1.0,268.0,0.0,1
2,2025-03-31,2025-04-29,2025-04-30,2025-05-29,140.0,1.0,261.0,0.0,1
3,2025-04-01,2025-04-30,2025-05-01,2025-05-30,146.0,1.0,257.0,0.0,1
4,2025-04-02,2025-05-01,2025-05-02,2025-05-31,146.0,1.0,257.0,0.0,1
5,2025-04-03,2025-05-02,2025-05-03,2025-06-01,157.0,1.0,262.0,0.0,1
6,2025-04-04,2025-05-03,2025-05-04,2025-06-02,164.0,1.0,258.0,0.0,1
7,2025-04-05,2025-05-04,2025-05-05,2025-06-03,169.0,0.0,265.0,0.0,0
8,2025-04-06,2025-05-05,2025-05-06,2025-06-04,183.0,0.0,267.0,0.0,0
9,2025-04-07,2025-05-06,2025-05-07,2025-06-05,182.0,0.0,252.0,0.0,0


## 12. Non-Overlapping Historical Windows

In [9]:
historical_windows_30d = con.sql(f"""
WITH daily AS (
    SELECT
        content_hash_id,
        report_date,
        gsc_impressions,
        gsc_clicks
    FROM {TABLES['fact_daily']}
    WHERE gsc_data_available = TRUE
),

date_bounds AS (
    SELECT
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM daily
),

window_starts AS (
    SELECT
        min_date + (30 * window_number) * INTERVAL 1 DAY AS observation_start
    FROM date_bounds,
    generate_series(
        0,
        CAST(
            DATE_DIFF('day', min_date, max_date) / 30
            AS INTEGER
        )
    ) AS t(window_number)
),

content_windows AS (
    SELECT
        c.content_hash_id,
        w.observation_start
    FROM (
        SELECT DISTINCT content_hash_id
        FROM daily
    ) c
    CROSS JOIN window_starts w
),

window_totals AS (
    SELECT
        cw.content_hash_id,
        cw.observation_start,

        SUM(
            CASE
                WHEN d.report_date >= cw.observation_start
                 AND d.report_date < cw.observation_start + INTERVAL 30 DAY
                THEN d.gsc_impressions
                ELSE 0
            END
        ) AS observation_impressions,

        SUM(
            CASE
                WHEN d.report_date >= cw.observation_start
                 AND d.report_date < cw.observation_start + INTERVAL 30 DAY
                THEN d.gsc_clicks
                ELSE 0
            END
        ) AS observation_clicks,

        SUM(
            CASE
                WHEN d.report_date >= cw.observation_start + INTERVAL 30 DAY
                 AND d.report_date < cw.observation_start + INTERVAL 60 DAY
                THEN d.gsc_impressions
                ELSE 0
            END
        ) AS outcome_impressions,

        SUM(
            CASE
                WHEN d.report_date >= cw.observation_start + INTERVAL 30 DAY
                 AND d.report_date < cw.observation_start + INTERVAL 60 DAY
                THEN d.gsc_clicks
                ELSE 0
            END
        ) AS outcome_clicks

    FROM content_windows cw
    LEFT JOIN daily d
        ON d.content_hash_id = cw.content_hash_id
       AND d.report_date >= cw.observation_start
       AND d.report_date < cw.observation_start + INTERVAL 60 DAY

    GROUP BY
        cw.content_hash_id,
        cw.observation_start
)

SELECT
    content_hash_id,

    observation_start,
    observation_start + INTERVAL 29 DAY AS observation_end,

    observation_start + INTERVAL 30 DAY AS outcome_start,
    observation_start + INTERVAL 59 DAY AS outcome_end,

    observation_impressions,
    observation_clicks,
    outcome_impressions,
    outcome_clicks,

    CASE
        WHEN observation_impressions >= 100
         AND outcome_clicks <= observation_clicks * 0.60
        THEN 1
        ELSE 0
    END AS is_declining_label

FROM window_totals

WHERE observation_impressions >= 100
  AND observation_start + INTERVAL 59 DAY <= (
      SELECT max_date
      FROM date_bounds
  )

ORDER BY
    content_hash_id,
    observation_start
""").df()

# Register the DataFrame as a DuckDB view so subsequent SQL queries can access it
con.register('historical_windows_30d', historical_windows_30d)

historical_windows_30d.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,observation_start,observation_end,outcome_start,outcome_end,observation_impressions,observation_clicks,outcome_impressions,outcome_clicks,is_declining_label
0,content_000005d4ced12088,2025-04-27,2025-05-26,2025-05-27,2025-06-25,264.0,0.0,171.0,0.0,1
1,content_000005d4ced12088,2025-05-27,2025-06-25,2025-06-26,2025-07-25,171.0,0.0,170.0,0.0,1
2,content_000005d4ced12088,2025-06-26,2025-07-25,2025-07-26,2025-08-24,170.0,0.0,583.0,1.0,0
3,content_000005d4ced12088,2025-07-26,2025-08-24,2025-08-25,2025-09-23,583.0,1.0,710.0,0.0,1
4,content_000005d4ced12088,2025-08-25,2025-09-23,2025-09-24,2025-10-23,710.0,0.0,247.0,1.0,0


In [33]:
print("Total non-overlapping windows:", len(historical_windows_30d))

print("\nTarget distribution:")
print(
    historical_windows_30d["is_declining_label"]
    .value_counts()
    .sort_index()
)

print("\nTarget distribution (%):")
print(
    historical_windows_30d["is_declining_label"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

Total non-overlapping windows: 651458

Target distribution:
is_declining_label
0    321436
1    330022
Name: count, dtype: int64

Target distribution (%):
is_declining_label
0    49.34
1    50.66
Name: proportion, dtype: float64


## 13. Target Validation

In [34]:
# Verify that every positive label satisfies the 40% decline rule
positive_cases = historical_windows_30d[
    historical_windows_30d["is_declining_label"] == 1
].copy()

positive_cases["click_change_pct"] = (
    (positive_cases["outcome_clicks"] - positive_cases["observation_clicks"])
    / positive_cases["observation_clicks"].replace(0, float("nan"))
) * 100

print("Positive labels:", len(positive_cases))

print("\nClick change among positive labels:")
print(positive_cases["click_change_pct"].describe())

Positive labels: 330022

Click change among positive labels:
count    152106.000000
mean        -81.222517
std          22.013095
min        -100.000000
25%        -100.000000
50%        -100.000000
75%         -60.000000
max         -40.000000
Name: click_change_pct, dtype: float64


In [35]:
# Check for any positive labels that do NOT represent at least a 40% decline

invalid_positive = historical_windows_30d[
    (historical_windows_30d["is_declining_label"] == 1)
    &
    (
        historical_windows_30d["outcome_clicks"]
        > historical_windows_30d["observation_clicks"] * 0.60
    )
]

print("Invalid positive labels:", len(invalid_positive))

Invalid positive labels: 0


## 14. Observation-Window Predictor Features

In [36]:
# Build model features using ONLY the 30-day observation window

model_data = historical_windows_30d.copy()

# Basic observation-window features
model_data["observation_ctr"] = (
    model_data["observation_clicks"]
    / model_data["observation_impressions"]
)

# Keep only information available at prediction time
feature_columns = [
    "observation_impressions",
    "observation_clicks",
    "observation_ctr"
]

X = model_data[feature_columns].copy()
y = model_data["is_declining_label"].copy()

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

print("\nFeatures:")
print(X.head())

print("\nMissing values:")
print(X.isna().sum())

Feature matrix shape: (651458, 3)
Target shape: (651458,)

Features:
   observation_impressions  observation_clicks  observation_ctr
0                    264.0                 0.0         0.000000
1                    171.0                 0.0         0.000000
2                    170.0                 0.0         0.000000
3                    583.0                 1.0         0.001715
4                    710.0                 0.0         0.000000

Missing values:
observation_impressions    0
observation_clicks         0
observation_ctr            0
dtype: int64


## 15. Expanded Observation-Window Features

In [37]:
print("Current model data:")
print("Rows:", len(model_data))
print("Features:", list(X.columns))

print("\nTarget balance:")
print(y.value_counts(normalize=True).mul(100).round(2))

Current model data:
Rows: 651458
Features: ['observation_impressions', 'observation_clicks', 'observation_ctr']

Target balance:
is_declining_label
1    50.66
0    49.34
Name: proportion, dtype: float64


## 16. Baseline Dataset

The baseline model uses only three observation-window signals:

- Total GSC impressions
- Total GSC clicks
- GSC click-through rate (CTR)

All features are calculated from information available during the 30-day observation window. The subsequent 30-day outcome window is used only to construct `is_declining_label`.

In [38]:
# Create the baseline dataset

baseline_data = model_data[
    feature_columns + ["is_declining_label"]
].copy()

print("Baseline dataset shape:", baseline_data.shape)

print("\nColumns:")
print(baseline_data.columns.tolist())

print("\nMissing values:")
print(baseline_data.isna().sum())

print("\nTarget distribution:")
print(
    baseline_data["is_declining_label"]
    .value_counts()
    .sort_index()
)

Baseline dataset shape: (651458, 4)

Columns:
['observation_impressions', 'observation_clicks', 'observation_ctr', 'is_declining_label']

Missing values:
observation_impressions    0
observation_clicks         0
observation_ctr            0
is_declining_label         0
dtype: int64

Target distribution:
is_declining_label
0    321436
1    330022
Name: count, dtype: int64


In [39]:
baseline_data.groupby("is_declining_label")[
    [
        "observation_impressions",
        "observation_clicks",
        "observation_ctr"
    ]
].mean().round(4)

,observation_impressions,observation_clicks,observation_ctr
is_declining_label,,,
0,3176.1262,11.2805,0.0038
1,1288.8049,3.7530,0.0023


## 17. Time-Based Train/Test Split

The dataset is split chronologically so that earlier observation windows are used for training and later observation windows are reserved for evaluation.

This reflects the real deployment scenario: train on historical content behavior and predict future content decline.

In [40]:
# Keep the temporal information for splitting

baseline_data = model_data[
    [
        "content_hash_id",
        "observation_start",
        "observation_end",
        "observation_impressions",
        "observation_clicks",
        "observation_ctr",
        "is_declining_label"
    ]
].copy()

baseline_data = baseline_data.sort_values(
    "observation_start"
).reset_index(drop=True)

print("Baseline dataset shape:", baseline_data.shape)
print(
    "Date range:",
    baseline_data["observation_start"].min(),
    "to",
    baseline_data["observation_start"].max()
)

Baseline dataset shape: (651458, 7)
Date range: 2025-01-27 00:00:00 to 2026-04-22 00:00:00


In [41]:
split_date = baseline_data["observation_start"].quantile(0.80)

train_data = baseline_data[
    baseline_data["observation_start"] < split_date
].copy()

test_data = baseline_data[
    baseline_data["observation_start"] >= split_date
].copy()

print("Split date:", split_date)

print("\nTraining rows:", len(train_data))
print("Testing rows:", len(test_data))

print("\nTraining date range:")
print(train_data["observation_start"].min(), "to", train_data["observation_start"].max())

print("\nTesting date range:")
print(test_data["observation_start"].min(), "to", test_data["observation_start"].max())

Split date: 2026-03-23 00:00:00

Training rows: 435906
Testing rows: 215552

Training date range:
2025-01-27 00:00:00 to 2026-02-21 00:00:00

Testing date range:
2026-03-23 00:00:00 to 2026-04-22 00:00:00


## 18. Baseline Logistic Regression

A logistic regression model is used as the first ML baseline.

The model predicts the probability that a content item will be labeled as declining using only three observation-window features:

- GSC impressions
- GSC clicks
- GSC CTR

The model is trained only on earlier observation windows and evaluated on later observation windows to preserve the temporal nature of the prediction task.

In [42]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

# Features and target
feature_columns = [
    "observation_impressions",
    "observation_clicks",
    "observation_ctr"
]

X_train = train_data[feature_columns]
y_train = train_data["is_declining_label"]

X_test = test_data[feature_columns]
y_test = test_data["is_declining_label"]

# Train baseline model
baseline_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

baseline_model.fit(X_train, y_train)

# Predictions
y_pred = baseline_model.predict(X_test)
y_prob = baseline_model.predict_proba(X_test)[:, 1]

# Evaluation
print("Baseline Logistic Regression")
print("-" * 40)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("Precision:", round(precision_score(y_test, y_pred), 4))
print("Recall:", round(recall_score(y_test, y_pred), 4))
print("F1:", round(f1_score(y_test, y_pred), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 4))
print("PR-AUC:", round(average_precision_score(y_test, y_prob), 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Baseline Logistic Regression
----------------------------------------
Accuracy: 0.6385
Precision: 0.6479
Recall: 0.7358
F1: 0.6891
ROC-AUC: 0.7022
PR-AUC: 0.7358

Confusion Matrix:
[[51286 46921]
 [30997 86348]]


## 19. Baseline Evaluation Verification

In [43]:
from sklearn.metrics import classification_report

print(classification_report(
    y_test,
    y_pred,
    target_names=["Not declining", "Declining"],
    digits=4
))

print("\nPrediction probabilities:")
print(y_prob[:10])

               precision    recall  f1-score   support

Not declining     0.6233    0.5222    0.5683     98207
    Declining     0.6479    0.7358    0.6891    117345

     accuracy                         0.6385    215552
    macro avg     0.6356    0.6290    0.6287    215552
 weighted avg     0.6367    0.6385    0.6341    215552


Prediction probabilities:
[0.56364098 0.51278768 0.51427222 0.57920703 0.52831773 0.5785538
 0.13485023 0.58131001 0.58348239 0.51567163]


In [44]:
print("Unique predicted classes:", sorted(set(y_pred)))
print("Probability range:", y_prob.min(), "to", y_prob.max())

print("\nROC-AUC:", round(roc_auc_score(y_test, y_prob), 4))
print("Average Precision / PR-AUC:", round(
    average_precision_score(y_test, y_prob), 4
))

Unique predicted classes: [np.int32(0), np.int32(1)]
Probability range: 5.336637821960905e-67 to 0.5836632776812203

ROC-AUC: 0.7022
Average Precision / PR-AUC: 0.7358


## 20. Expanded Observation-Window Features

The model will use additional signals calculated exclusively from the 30-day observation window.

No outcome-window variables will be used as predictors to prevent temporal leakage.

In [10]:
gsc_features = con.sql(f"""
SELECT
    h.content_hash_id,
    h.observation_start,
    h.observation_end,
    h.is_declining_label,

    SUM(d.gsc_impressions) AS impressions,
    SUM(d.gsc_clicks) AS clicks,

    CASE
        WHEN SUM(d.gsc_impressions) > 0
        THEN SUM(d.gsc_clicks) * 1.0 / SUM(d.gsc_impressions)
        ELSE 0
    END AS ctr,

    AVG(d.gsc_avg_position) AS avg_position

FROM historical_windows_30d h

JOIN {TABLES['fact_daily']} d
    ON d.content_hash_id = h.content_hash_id
    AND d.report_date >= h.observation_start
    AND d.report_date <= h.observation_end
    AND d.gsc_data_available = TRUE

GROUP BY
    h.content_hash_id,
    h.observation_start,
    h.observation_end,
    h.is_declining_label
""").df()

print("Shape:", gsc_features.shape)
gsc_features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Shape: (651458, 8)


,content_hash_id,observation_start,observation_end,is_declining_label,impressions,clicks,ctr,avg_position
0,content_28cdb373426c7008,2025-07-26,2025-08-24,1,194.0,0.0,0.000000,56.940930
1,content_28d09f8323dc105d,2025-11-23,2025-12-22,1,422.0,0.0,0.000000,7.524804
2,content_28d48fb09c4a0301,2026-03-23,2026-04-21,1,120.0,0.0,0.000000,5.500795
3,content_28fc0602c6833107,2026-01-22,2026-02-20,1,257.0,1.0,0.003891,23.884761
4,content_2907748b47398527,2026-04-22,2026-05-21,1,11030.0,46.0,0.004170,5.411900


In [12]:
ga4_features = con.sql(f"""
SELECT
    h.content_hash_id,
    h.observation_start,
    h.observation_end,

    SUM(d.ga4_pageviews) AS pageviews,
    SUM(d.ga4_sessions) AS sessions,
    SUM(d.ga4_users) AS users,
    SUM(d.ga4_engaged_sessions) AS engaged_sessions,
    SUM(d.ga4_total_engagement_sec) AS engagement_seconds

FROM historical_windows_30d h

JOIN {TABLES['fact_daily']} d
    ON d.content_hash_id = h.content_hash_id
    AND d.report_date >= h.observation_start
    AND d.report_date <= h.observation_end
    AND d.ga4_data_available = TRUE

GROUP BY
    h.content_hash_id,
    h.observation_start,
    h.observation_end
""").df()

print("Shape:", ga4_features.shape)
ga4_features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Shape: (237035, 8)


,content_hash_id,observation_start,observation_end,pageviews,sessions,users,engaged_sessions,engagement_seconds
0,content_28d6e2fd3e57b579,2025-12-23,2026-01-21,2.0,2.0,2.0,0.0,0.0
1,content_28d7ed1458f51192,2026-03-23,2026-04-21,4.0,4.0,4.0,1.0,99.0
2,content_28ddf7fa4fede0ca,2026-04-22,2026-05-21,8.0,4.0,3.0,0.0,1.0
3,content_28e0ea1c1ab69597,2026-04-22,2026-05-21,7.0,4.0,4.0,0.0,0.0
4,content_28f5f29743f3b0fa,2026-04-22,2026-05-21,110.0,58.0,57.0,2.0,333.0


In [13]:
print(
    "Memory:",
    ga4_features.memory_usage(deep=True).sum() / 1024**2,
    "MB"
)

print("\nMissing values:")
print(ga4_features.isna().sum())

Memory: 29.16111660003662 MB

Missing values:
content_hash_id       0
observation_start     0
observation_end       0
pageviews             0
sessions              0
users                 0
engaged_sessions      0
engagement_seconds    0
dtype: int64


In [14]:
expanded_features = gsc_features.merge(
    ga4_features,
    on=[
        "content_hash_id",
        "observation_start",
        "observation_end"
    ],
    how="left"
)

print("Shape:", expanded_features.shape)
print("\nMissing values:")
print(expanded_features.isna().sum())

Shape: (651458, 13)

Missing values:
content_hash_id            0
observation_start          0
observation_end            0
is_declining_label         0
impressions                0
clicks                     0
ctr                        0
avg_position               0
pageviews             414423
sessions              414423
users                 414423
engaged_sessions      414423
engagement_seconds    414423
dtype: int64


In [15]:
ga4_columns = [
    "pageviews",
    "sessions",
    "users",
    "engaged_sessions",
    "engagement_seconds"
]

expanded_features[ga4_columns] = (
    expanded_features[ga4_columns]
    .fillna(0)
)

print("Final shape:", expanded_features.shape)
print("\nMissing values after filling:")
print(expanded_features.isna().sum())

Final shape: (651458, 13)

Missing values after filling:
content_hash_id       0
observation_start     0
observation_end       0
is_declining_label    0
impressions           0
clicks                0
ctr                   0
avg_position          0
pageviews             0
sessions              0
users                 0
engaged_sessions      0
engagement_seconds    0
dtype: int64


###Models and their comparison

In [17]:
import pandas as pd

# Same chronological split used for the baseline model
split_date = pd.Timestamp("2026-03-23")

expanded_train = expanded_features[
    expanded_features["observation_start"] < split_date
].copy()

expanded_test = expanded_features[
    expanded_features["observation_start"] >= split_date
].copy()

print("Training shape:", expanded_train.shape)
print("Testing shape:", expanded_test.shape)

print("\nTraining date range:")
print(
    expanded_train["observation_start"].min(),
    "to",
    expanded_train["observation_start"].max()
)

print("\nTesting date range:")
print(
    expanded_test["observation_start"].min(),
    "to",
    expanded_test["observation_start"].max()
)

Training shape: (435906, 13)
Testing shape: (215552, 13)

Training date range:
2025-01-27 00:00:00 to 2026-02-21 00:00:00

Testing date range:
2026-03-23 00:00:00 to 2026-04-22 00:00:00


In [18]:
print("Training target distribution:")
print(
    expanded_train["is_declining_label"]
    .value_counts(normalize=True)
)

print("\nTesting target distribution:")
print(
    expanded_test["is_declining_label"]
    .value_counts(normalize=True)
)

Training target distribution:
is_declining_label
0    0.512104
1    0.487896
Name: proportion, dtype: float64

Testing target distribution:
is_declining_label
1    0.544393
0    0.455607
Name: proportion, dtype: float64


In [19]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

expanded_feature_columns = [
    "impressions",
    "clicks",
    "ctr",
    "avg_position",
    "pageviews",
    "sessions",
    "users",
    "engaged_sessions",
    "engagement_seconds"
]

X_train_expanded = expanded_train[expanded_feature_columns]
y_train_expanded = expanded_train["is_declining_label"]

X_test_expanded = expanded_test[expanded_feature_columns]
y_test_expanded = expanded_test["is_declining_label"]

expanded_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

expanded_model.fit(
    X_train_expanded,
    y_train_expanded
)

expanded_pred = expanded_model.predict(X_test_expanded)
expanded_prob = expanded_model.predict_proba(X_test_expanded)[:, 1]

print("Expanded Logistic Regression Results")
print("-" * 40)

print("Accuracy :", round(
    accuracy_score(y_test_expanded, expanded_pred), 4
))

print("Precision:", round(
    precision_score(y_test_expanded, expanded_pred), 4
))

print("Recall   :", round(
    recall_score(y_test_expanded, expanded_pred), 4
))

print("F1 Score :", round(
    f1_score(y_test_expanded, expanded_pred), 4
))

print("ROC-AUC  :", round(
    roc_auc_score(y_test_expanded, expanded_prob), 4
))

print("PR-AUC   :", round(
    average_precision_score(y_test_expanded, expanded_prob), 4
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_expanded, expanded_pred))

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Expanded Logistic Regression Results
----------------------------------------
Accuracy : 0.6058
Precision: 0.6559
Recall   : 0.5804
F1 Score : 0.6159
ROC-AUC  : 0.6607
PR-AUC   : 0.7

Confusion Matrix:
[[62476 35731]
 [49236 68109]]


In [20]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

scaled_model = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic", LogisticRegression(
        max_iter=2000,
        random_state=42
    ))
])

scaled_model.fit(
    X_train_expanded,
    y_train_expanded
)

scaled_pred = scaled_model.predict(X_test_expanded)
scaled_prob = scaled_model.predict_proba(X_test_expanded)[:, 1]

In [21]:
print("Scaled Expanded Logistic Regression")
print("-" * 45)

print("Accuracy :", round(
    accuracy_score(y_test_expanded, scaled_pred), 4
))

print("Precision:", round(
    precision_score(y_test_expanded, scaled_pred), 4
))

print("Recall   :", round(
    recall_score(y_test_expanded, scaled_pred), 4
))

print("F1 Score :", round(
    f1_score(y_test_expanded, scaled_pred), 4
))

print("ROC-AUC  :", round(
    roc_auc_score(y_test_expanded, scaled_prob), 4
))

print("PR-AUC   :", round(
    average_precision_score(y_test_expanded, scaled_prob), 4
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_expanded, scaled_pred))

Scaled Expanded Logistic Regression
---------------------------------------------
Accuracy : 0.626
Precision: 0.6649
Recall   : 0.6312
F1 Score : 0.6476
ROC-AUC  : 0.6713
PR-AUC   : 0.7122

Confusion Matrix:
[[60883 37324]
 [43282 74063]]


In [22]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    min_samples_leaf=20,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(
    X_train_expanded,
    y_train_expanded
)

rf_pred = rf_model.predict(X_test_expanded)
rf_prob = rf_model.predict_proba(X_test_expanded)[:, 1]

In [23]:
print("Random Forest Results")
print("-" * 40)

print("Accuracy :", round(
    accuracy_score(y_test_expanded, rf_pred), 4
))

print("Precision:", round(
    precision_score(y_test_expanded, rf_pred), 4
))

print("Recall   :", round(
    recall_score(y_test_expanded, rf_pred), 4
))

print("F1 Score :", round(
    f1_score(y_test_expanded, rf_pred), 4
))

print("ROC-AUC  :", round(
    roc_auc_score(y_test_expanded, rf_prob), 4
))

print("PR-AUC   :", round(
    average_precision_score(y_test_expanded, rf_prob), 4
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_expanded, rf_pred))

Random Forest Results
----------------------------------------
Accuracy : 0.6646
Precision: 0.7192
Recall   : 0.6298
F1 Score : 0.6715
ROC-AUC  : 0.7164
PR-AUC   : 0.7582

Confusion Matrix:
[[69348 28859]
 [43442 73903]]


In [24]:
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

nn_model = Pipeline([
    ("scaler", StandardScaler()),

    ("mlp", MLPClassifier(
        hidden_layer_sizes=(64, 32, 16),
        activation="relu",
        solver="adam",
        alpha=0.0001,
        batch_size=256,
        learning_rate_init=0.001,
        max_iter=50,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=5,
        random_state=42
    ))
])

nn_model.fit(
    X_train_expanded,
    y_train_expanded
)

nn_pred = nn_model.predict(X_test_expanded)
nn_prob = nn_model.predict_proba(X_test_expanded)[:, 1]

In [25]:
print("Neural Network Results")
print("-" * 40)

print("Accuracy :", round(
    accuracy_score(y_test_expanded, nn_pred), 4
))

print("Precision:", round(
    precision_score(y_test_expanded, nn_pred), 4
))

print("Recall   :", round(
    recall_score(y_test_expanded, nn_pred), 4
))

print("F1 Score :", round(
    f1_score(y_test_expanded, nn_pred), 4
))

print("ROC-AUC  :", round(
    roc_auc_score(y_test_expanded, nn_prob), 4
))

print("PR-AUC   :", round(
    average_precision_score(y_test_expanded, nn_prob), 4
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_expanded, nn_pred))

Neural Network Results
----------------------------------------
Accuracy : 0.6588
Precision: 0.7203
Recall   : 0.6101
F1 Score : 0.6607
ROC-AUC  : 0.7082
PR-AUC   : 0.7537

Confusion Matrix:
[[70411 27796]
 [45752 71593]]


**21 Tune the Random Forest threshold**

In [27]:
import numpy as np

threshold_results = []

for threshold in np.arange(0.30, 0.61, 0.05):

    threshold_pred = (rf_prob >= threshold).astype(int)

    threshold_results.append({
        "threshold": round(threshold, 2),
        "accuracy": accuracy_score(
            y_test_expanded,
            threshold_pred
        ),
        "precision": precision_score(
            y_test_expanded,
            threshold_pred,
            zero_division=0
        ),
        "recall": recall_score(
            y_test_expanded,
            threshold_pred,
            zero_division=0
        ),
        "f1": f1_score(
            y_test_expanded,
            threshold_pred,
            zero_division=0
        )
    })

threshold_results = pd.DataFrame(threshold_results)

print(threshold_results.round(4).to_string(index=False))

 threshold  accuracy  precision  recall     f1
      0.30    0.5905     0.5776  0.9219 0.7102
      0.35    0.6298     0.6164  0.8474 0.7137
      0.40    0.6551     0.6569  0.7671 0.7078
      0.45    0.6644     0.6898  0.6967 0.6933
      0.50    0.6646     0.7192  0.6298 0.6715
      0.55    0.6552     0.7466  0.5551 0.6368
      0.60    0.6415     0.7713  0.4853 0.5958


In [28]:
feature_importance = pd.DataFrame({
    "feature": expanded_feature_columns,
    "importance": rf_model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

print(feature_importance.to_string(index=False))

           feature  importance
               ctr    0.297127
            clicks    0.244410
       impressions    0.232074
      avg_position    0.176395
         pageviews    0.016783
          sessions    0.010622
             users    0.009973
engagement_seconds    0.009339
  engaged_sessions    0.003278


**Step 22 GSC-only Random Forest**

In [29]:
# GSC-only features
gsc_feature_columns = [
    "impressions",
    "clicks",
    "ctr",
    "avg_position"
]

X_train_gsc = expanded_train[gsc_feature_columns]
y_train_gsc = expanded_train["is_declining_label"]

X_test_gsc = expanded_test[gsc_feature_columns]
y_test_gsc = expanded_test["is_declining_label"]

gsc_rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    min_samples_leaf=20,
    random_state=42,
    n_jobs=-1
)

gsc_rf_model.fit(
    X_train_gsc,
    y_train_gsc
)

gsc_rf_pred = gsc_rf_model.predict(X_test_gsc)
gsc_rf_prob = gsc_rf_model.predict_proba(X_test_gsc)[:, 1]

In [30]:
print("GSC-only Random Forest Results")
print("-" * 45)

print("Accuracy :", round(
    accuracy_score(y_test_gsc, gsc_rf_pred), 4
))

print("Precision:", round(
    precision_score(y_test_gsc, gsc_rf_pred), 4
))

print("Recall   :", round(
    recall_score(y_test_gsc, gsc_rf_pred), 4
))

print("F1 Score :", round(
    f1_score(y_test_gsc, gsc_rf_pred), 4
))

print("ROC-AUC  :", round(
    roc_auc_score(y_test_gsc, gsc_rf_prob), 4
))

print("PR-AUC   :", round(
    average_precision_score(y_test_gsc, gsc_rf_prob), 4
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_gsc, gsc_rf_pred))

GSC-only Random Forest Results
---------------------------------------------
Accuracy : 0.6666
Precision: 0.7207
Recall   : 0.6328
F1 Score : 0.6739
ROC-AUC  : 0.722
PR-AUC   : 0.7615

Confusion Matrix:
[[69427 28780]
 [43086 74259]]


In [31]:
# Threshold tuning for GSC-only Random Forest

thresholds = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]

threshold_results = []

for threshold in thresholds:

    predictions = (gsc_rf_prob >= threshold).astype(int)

    threshold_results.append({
        "threshold": threshold,
        "accuracy": accuracy_score(y_test_gsc, predictions),
        "precision": precision_score(y_test_gsc, predictions),
        "recall": recall_score(y_test_gsc, predictions),
        "f1": f1_score(y_test_gsc, predictions)
    })

threshold_df = pd.DataFrame(threshold_results)

print(threshold_df.round(4).to_string(index=False))

 threshold  accuracy  precision  recall     f1
      0.30    0.6247     0.6078  0.8761 0.7177
      0.35    0.6474     0.6385  0.8122 0.7149
      0.40    0.6603     0.6673  0.7501 0.7062
      0.45    0.6666     0.6943  0.6924 0.6933
      0.50    0.6666     0.7207  0.6328 0.6739
      0.55    0.6594     0.7476  0.5651 0.6436
      0.60    0.6440     0.7708  0.4926 0.6011


In [32]:
# Final model evaluation at threshold = 0.40

final_threshold = 0.40

final_pred = (gsc_rf_prob >= final_threshold).astype(int)

cm = confusion_matrix(y_test_gsc, final_pred)

tn, fp, fn, tp = cm.ravel()

print("Final Model — Threshold 0.40")
print("-" * 45)

print("True Negatives :", tn)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Positives  :", tp)

print("\nTotal test cases:", len(y_test_gsc))

print("\nMetrics:")
print("Accuracy :", round(accuracy_score(y_test_gsc, final_pred), 4))
print("Precision:", round(precision_score(y_test_gsc, final_pred), 4))
print("Recall   :", round(recall_score(y_test_gsc, final_pred), 4))
print("F1 Score :", round(f1_score(y_test_gsc, final_pred), 4))

Final Model — Threshold 0.40
---------------------------------------------
True Negatives : 54316
False Positives: 43891
False Negatives: 29330
True Positives  : 88015

Total test cases: 215552

Metrics:
Accuracy : 0.6603
Precision: 0.6673
Recall   : 0.7501
F1 Score : 0.7062


**Final Step**

In [33]:
# Feature importance of the final GSC-only Random Forest

final_feature_importance = pd.DataFrame({
    "feature": gsc_feature_columns,
    "importance": gsc_rf_model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

print(final_feature_importance)

        feature  importance
2           ctr    0.308089
1        clicks    0.290437
0   impressions    0.218238
3  avg_position    0.183237
